# 1. Objective
  The Objective of the notebook is to process the outputs of LASSO algorithm for all the iterations and select the finalized iteration for each F_CODE based on pre-defined criteria (wherever applicable). It performs the following QCs
  - Whether the train & test WMAPE is within threshold
  - Whether overfitting is seen based on the threshold
  - Whether the WMAPE calculated on non-promo weeks is within the threshold

For each F_CODE, the model which pass all the above criteria are used as the final model. For the rest of the F_CODEs where the above criteria failed for all iterations, final model is selected based on the test WMAPE

# 2. Imports

In [ ]:
from datetime import datetime

import os
import copy
import numpy as np
import pandas as pd
import sys
import traceback
import shutil

# --- Snowpark Imports ---
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import col
from snowflake.snowpark.window import Window
from snowflake.snowpark.types import IntegerType, StringType, FloatType, StructField

# ======================================================
# Initialize Snowpark Session
# ======================================================

# If running inside Snowflake (e.g., Snowflake Worksheet, Snowsight, or Streamlit for Snowflake)
session = get_active_session()

# If running locally, uncomment and configure connection details:

# ======================================================
# Snowpark is now ready to use (Equivalent to SparkSession)
# ======================================================

# Example check (optional)
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 3. Setup environment

## 3.1. Load Config

In [ ]:
import yaml
stage_path = "@ORANGE_ZONE_SBX_TA.PUBLIC.CONNECTIONS/config_new_PROD.yaml"
stream = session.file.get_stream(stage_path)
yaml_text = stream.read().decode()
app_config = yaml.safe_load(yaml_text)

## 3.2. Update Output Database, Schema , table

In [ ]:
output_database = app_config["general_inputs"]["output_database"]
output_schema = app_config["general_inputs"]["output_schema"]
print(output_database, output_schema)

In [ ]:
session.use_database(output_database)
session.use_schema(output_schema)
coef_output_table_name = "PROD_LASSO_COEFFICIENTS_FINALIZED_ITER"
cont_output_table_name = "PROD_LASSO_CONTRIBUTIONS_FINALIZED_ITER"
final_output_table_name = "PROD_FINAL_MODEL_OUTPUT"
intermediate_table_name = "PROD_LASSO_QC_RESULTS_INTERMEDIATE_OUTPUT"

In [ ]:
# Example check (optional)
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 4. Utilities

## 4.1. Broadcasting API

In [0]:
# Function to broadcast variables to be used in UDF
def broadcast_variable_conf(x):
    """Function to broadcast the required variables from config file

    Parameters
    ----------
    x : _type_
        The required variables from config file

    Returns
    -------
    spark.sparkContext.broadcast
        Returns the broadcasted version of the required variables from config file
    """
    #x = spark.sparkContext.broadcast(x)
    return x

## 4.2. Overall Rsq, WMAPE Check - API

In [0]:
def train_test_rsq_mape_check(df, rsq_cutoff, mapecutoff, overfit_cutoff):
    df['train_rsq_flag'] = np.where(df['TRAIN_RSQ'] < rsq_cutoff, 1, 0)
    df['test_rsq_flag'] = np.where(df['TEST_RSQ'] < rsq_cutoff, 1, 0)
    df['train_wmape_flag'] = np.where(df['TRAIN_WMAPE'] > mapecutoff, 1, 0)
    df['test_wmape_flag'] = np.where(df['TEST_WMAPE'] > mapecutoff, 1, 0)

    df['overfitting_model'] = np.where(df['TRAIN_WMAPE'] - df['TEST_WMAPE'] > overfit_cutoff, 1, 0)
    return df

In [0]:
def rsq_mape_check(df, rsq_cutoff, mapecutoff, overfit_cutoff):
    df['rsq_flag'] = np.where(df['rsq'] < rsq_cutoff, 1, 0)
    df['wmape_flag'] = np.where(df['wmape'] > mapecutoff, 1, 0)

    return df

## 4.3. Deviation Flags - API

In [0]:
def baseline_checks_act_pred(df, actual_col, predicted_base_col, tpr_col, no_of_inc_base_cutoff):
    act_pred_base_df = df[df[tpr_col] == 0]
    act_pred_base_df = act_pred_base_df[act_pred_base_df['OUTLIER'] == False]
    no_of_weeks = act_pred_base_df.shape[0]
    act_pred_base_df['incorrect_baseline'] = np.where(((act_pred_base_df[predicted_base_col] > 1.3*act_pred_base_df[actual_col])), 1, 0)
    
    no_of_inc_base_df = act_pred_base_df[act_pred_base_df['incorrect_baseline'] == 1].groupby(['ITERATION_ID'], as_index = False)['START_OF_WEEK'].count()
    no_of_inc_base_df.rename(columns = {'START_OF_WEEK':'no_of_incorrect_base'}, inplace = True)
    
    df = df.merge(no_of_inc_base_df, on = ['ITERATION_ID'], how = 'left')
    
    no_of_inc_base_cutoff = np.round(no_of_weeks*0.1)
    
    df['cut_off_no_of_weeks'] = no_of_inc_base_cutoff
    df['too_many_incorrect_base'] = np.where(df['no_of_incorrect_base'] > no_of_inc_base_cutoff, 1, 0)

    return df

## 4.4. Baseline WMAPE - API

In [0]:
def baseline_check_mape(df, actual_col, predicted_base_col, tpr_col, baseline_mape_cutoff):
    pred_base_df =  df[df[tpr_col] == 0]
    
    pred_base_df = pred_base_df[pred_base_df['OUTLIER'] == False]
    pred_base_df['base_diff'] = np.abs(pred_base_df[predicted_base_col] - pred_base_df[actual_col])*pred_base_df[actual_col]
    pred_base_df['a2'] = pred_base_df[actual_col]*pred_base_df[actual_col]

    mape_df = pred_base_df.groupby([*modeling_granularity,'ITERATION_ID'])[['base_diff','a2']].sum().reset_index()
    mape_df['base_MAPE'] = mape_df['base_diff']/(mape_df['a2'])
    
    high_base_MAPE_df = mape_df[['ITERATION_ID','base_MAPE']].drop_duplicates()
    high_base_MAPE_df['high_base_MAPE'] = np.where(high_base_MAPE_df['base_MAPE'] > baseline_mape_cutoff, 1, 0)

    df = pd.merge(df, high_base_MAPE_df, on = ['ITERATION_ID'], how = 'left')
    return df

# 5. Broadcast generic parameters

In [ ]:
date_variable = app_config["general_inputs"]["date_var"]
date_format = app_config["general_inputs"]["date_format_pyspark"]
date_format_pandas = app_config["general_inputs"]["date_format_pandas"]

modeling_granularity = app_config["general_inputs"]["modeling_granularity"]

train_test_rsq_mape_check_needed = app_config["qc_checks"]['train_test_rsq_mape_check_needed']
rsq_mape_check_needed = app_config["qc_checks"]['rsq_mape_check_needed']
baseline_checks_act_pred_needed = app_config["qc_checks"]['baseline_checks_act_pred_needed']
baseline_check_mape_needed = app_config["qc_checks"]['baseline_check_mape_needed']
corr_base_act_check_needed = app_config["qc_checks"]['corr_base_act_check_needed']
variance_in_base_check_needed = app_config["qc_checks"]['variance_in_base_check_needed']
base_trends_check_needed = app_config["qc_checks"]['base_trends_check_needed']

rsq_cutoff = app_config["qc_checks"]['rsq_cutoff']
mapecutoff = app_config["qc_checks"]['mapecutoff']
overfit_cutoff = app_config["qc_checks"]['overfit_cutoff']

no_of_inc_base_cutoff = app_config['qc_checks']['no_of_inc_base_cutoff']
baseline_mape_cutoff = app_config["qc_checks"]['baseline_mape_cutoff']
corr_cut_off = app_config["qc_checks"]['corr_cut_off']
var_cut_off = app_config["qc_checks"]['var_cut_off']
trend_slope_cut_off = app_config["qc_checks"]['trend_slope_cut_off']

actual_col = app_config['qc_checks']['actual_col']
predicted_base_col = app_config['qc_checks']['predicted_base_col']
tpr_col = app_config['qc_checks']['tpr_col']

latest_q_base_check_needed = app_config['qc_checks']['latest_q_base_check_needed']

In [0]:
broadcast_tpr_col = broadcast_variable_conf(tpr_col)
broadcast_predicted_base_col = broadcast_variable_conf(predicted_base_col)
broadcast_actual_col = broadcast_variable_conf(actual_col)

broadcast_date_variable = broadcast_variable_conf(date_variable)
broadcast_modeling_granularity = broadcast_variable_conf(modeling_granularity)
broadcast_rsq_mape_check_needed = broadcast_variable_conf(rsq_mape_check_needed)
broadcast_train_test_rsq_mape_check_needed = broadcast_variable_conf(train_test_rsq_mape_check_needed)
broadcast_baseline_checks_act_pred_needed = broadcast_variable_conf(baseline_checks_act_pred_needed)
broadcast_baseline_check_mape_needed = broadcast_variable_conf(baseline_check_mape_needed)
broadcast_corr_base_act_check_needed = broadcast_variable_conf(corr_base_act_check_needed)
broadcast_variance_in_base_check_needed = broadcast_variable_conf(variance_in_base_check_needed)
broadcast_base_trends_check_needed = broadcast_variable_conf(base_trends_check_needed)

broadcast_rsq_cutoff = broadcast_variable_conf(rsq_cutoff)
broadcast_mapecutoff = broadcast_variable_conf(mapecutoff)
broadcast_overfit_cutoff = broadcast_variable_conf(overfit_cutoff)

broadcast_no_of_inc_base_cutoff = broadcast_variable_conf(no_of_inc_base_cutoff)

broadcast_baseline_mape_cutoff = broadcast_variable_conf(baseline_mape_cutoff)
broadcast_corr_cut_off = broadcast_variable_conf(corr_cut_off)
broadcast_var_cut_off = broadcast_variable_conf(var_cut_off)
broadcast_trend_slope_cut_off = broadcast_variable_conf(trend_slope_cut_off)

broadcast_latest_q_base_check_needed = broadcast_variable_conf(latest_q_base_check_needed)

# 6. QC Module

## 6.1. Load model data

In [0]:
df_data = session.table("PROD_FEATURE_ENGINEERING_OUTPUT")

In [ ]:
df_data

## 6.2. Load LASSO results

In [ ]:
contrib_data = session.table("PROD_LASSO_CONTRIBUTIONS_ALL_ITERS")
contrib_data = contrib_data.fillna(0)
# Rename columns
contrib_data = contrib_data.withColumnRenamed("DS", date_variable)
contrib_data

In [ ]:
#from snowflake.snowpark.functions import col, substr, lit, split
contrib_data = contrib_data.with_column(
    "REGIONNAME",
    F.substr(F.col("ITERATION_ID"), 1, 5)
)
contrib_data = contrib_data.with_column(
    "F_CODE",
    F.substr(F.col("ITERATION_ID"), 7, 4)
)
contrib_data

In [ ]:
for col in modeling_granularity:
    contrib_data = contrib_data.withColumn(col, F.col(col).cast("string"))

In [0]:
model_coeff_df = session.table("PROD_LASSO_COEFFICIENTS_ALL_ITERS")

# Cast modeling_granularity to string
for col in modeling_granularity:
    model_coeff_df = model_coeff_df.withColumn(col, F.col(col).cast("string"))

# Rename columns
model_coeff_df = model_coeff_df.withColumnRenamed("rsq", "train_rsq") \
                               .withColumnRenamed("wmape", "train_wmape")

In [0]:
model_coeff_df

## 6.3. Detect outliers

In [ ]:
#actual_col = "NO_OF_NEW_JOINEES"

In [0]:
# Step 1: Filter rows where TPR_finalized == 0
df_filtered = df_data[df_data[tpr_col] == 0]


def create_schema_for_outliers(string_cols = [], float_cols = [], date_cols = []) -> StructType:
    """Function to create the schema of future baseline sales output.

    Returns
    -------
    StructType
        Returns the desired schema.
    """
    
    output_data_schema = []
                
    for x in string_cols:
        output_data_schema.append(StructField(x,StringType()))
    
    for x in date_cols:
        output_data_schema.append(StructField(x,DateType()))
    
    #other_float_cols = ["Week","Month",si_weekly,si_monthly,forecast_column_name]
    for x in float_cols:
        output_data_schema.append(StructField(x,FloatType()))
    
    return StructType(output_data_schema)

# Step 2: Group by 'Product UPC' and 'POSWeekStartDate'
def identify_outliers(df: pd.DataFrame) -> pd.DataFrame:
    
    Q1 = df[actual_col].quantile(0.25)  # 25th percentile
    Q3 = df[actual_col].quantile(0.75)  # 75th percentile
    IQR = Q3 - Q1  # Interquartile Range
    
    # Calculate bounds for outliers
    lower_bound = Q1 - IQR
    upper_bound = Q3 + IQR
    if df['F_CODE'].values[0] == '0307':
        print("Q1",Q1,"Q3",Q3,"IQR",IQR,"lower_bound",lower_bound,"upper_bound",upper_bound)
    
    # Mark outliers
    df['outlier'] = (df[actual_col] < lower_bound) | (df[actual_col] > upper_bound)
    
    return df[[*modeling_granularity, date_variable, "outlier"]]

# Step 3: Apply the outlier identification for each group
df_filtered = df_filtered[[*modeling_granularity, date_variable, actual_col]].group_by(modeling_granularity).applyInPandas(identify_outliers, output_schema = create_schema_for_outliers(string_cols = modeling_granularity, float_cols = ["outlier"], date_cols = [date_variable]))

# Step 4: Merge the results back to the original DataFrame
df_data = df_data.join(df_filtered[[*modeling_granularity, date_variable, 'outlier']], on=[*modeling_granularity, date_variable], how='left')

In [0]:
df_data

In [0]:
if "ITERATION_ID" not in model_coeff_df.columns:
    model_coeff_df = model_coeff_df.withColumn(
    "ITERATION_ID",
    F.concat_ws(F.lit("_"), *[F.col(c).cast("string") for c in modeling_granularity + ["alpha", "lambda"]])
)

In [ ]:
# 2. Select rsq/mape columns and drop duplicates
rsq_mape_df = model_coeff_df.select(
   ["ITERATION_ID", "TEST_RSQ", "TEST_WMAPE", "TRAIN_RSQ", "TRAIN_WMAPE"]
).dropDuplicates()

# 3. Cast rsq/mape columns to float
rsq_mape_df = rsq_mape_df.withColumn("TEST_RSQ", F.col("TEST_RSQ").cast("double")) \
                         .withColumn("TEST_WMAPE", F.col("TEST_WMAPE").cast("double")) \
                         .withColumn("TRAIN_RSQ", F.col("TRAIN_RSQ").cast("double")) \
                         .withColumn("TRAIN_WMAPE", F.col("TRAIN_WMAPE").cast("double"))

# 4. Join contrib_data with rsq_mape_df
contrib_data = contrib_data.join(rsq_mape_df, on=["ITERATION_ID"], how="inner")

# 5. Convert POSWeekStartDate to timestamp
contrib_data = contrib_data.withColumn(date_variable, F.to_timestamp(date_variable))
df_data = df_data.withColumn(date_variable, F.to_timestamp(date_variable))

In [ ]:
contrib_data

In [0]:
# 7. Join contrib_data with sim_data (left join on UPC + date)
sim_data_sel = df_data.select(
    *modeling_granularity, date_variable,tpr_col, actual_col, "OUTLIER"
).dropDuplicates()

contrib_data = contrib_data.join(
    sim_data_sel,
    on=[*modeling_granularity, date_variable],
    how="left"
)

contrib_data

In [ ]:
sim_data_sel

In [0]:
contrib_data.select('F_CODE').distinct().count()

In [0]:
#contrib_data.display()

## 6.4. Schema for UDF

In [0]:
input_data_schema = [x for x in contrib_data.schema]
output_data_schema = [] + input_data_schema


if train_test_rsq_mape_check_needed:
    broadcast_train_test_rsq_mape_check_schema = [
        StructField("train_rsq_flag",FloatType()),
        StructField("test_rsq_flag",FloatType()),
        StructField("train_wmape_flag",FloatType()),
        StructField("test_wmape_flag",FloatType()),
        StructField("overfitting_model",FloatType()),
    ]
    output_data_schema = output_data_schema + broadcast_train_test_rsq_mape_check_schema

if rsq_mape_check_needed:
    rsq_mape_check_schema = [
        StructField("rsq_flag",FloatType()),
        StructField("wmape_flag",FloatType()),
    ]
    output_data_schema = output_data_schema + rsq_mape_check_schema



if baseline_checks_act_pred_needed:

    baseline_checks_act_pred_schema = [
        StructField("no_of_incorrect_base",FloatType()),
        StructField('cut_off_no_of_weeks',FloatType()),
        StructField("too_many_incorrect_base",FloatType()),        
    ]
            
    output_data_schema = output_data_schema + baseline_checks_act_pred_schema

if baseline_check_mape_needed:

    baseline_check_mape_schema = [
        StructField("base_MAPE",FloatType()),
        StructField("high_base_MAPE",FloatType()),
    ]
            
    output_data_schema = output_data_schema + baseline_check_mape_schema

output_data_schema = output_data_schema + [StructField("status", StringType())]
output_data_schema = StructType(output_data_schema)

print(output_data_schema)

## 6.5. UDF for QC Checks

In [0]:
def qc_modules_udf(udf_input_data: pd.DataFrame) -> pd.DataFrame:
    try:    
        udf_output_data = udf_input_data.copy()

        # Get the parameter values from the broadcasted variables
        date_column = broadcast_date_variable

        # if there is train and test data split, then use this
        if broadcast_train_test_rsq_mape_check_needed:
            udf_output_data = train_test_rsq_mape_check(
                df = udf_output_data, 
                rsq_cutoff = broadcast_rsq_cutoff, 
                mapecutoff = broadcast_mapecutoff, 
                overfit_cutoff = broadcast_overfit_cutoff,
            )
        
        # if there is no train - test split use this
        if broadcast_rsq_mape_check_needed:
            udf_output_data = rsq_mape_check(
                df = udf_output_data, 
                rsq_cutoff = broadcast_rsq_cutoff, 
                mapecutoff = broadcast_mapecutoff, 
                overfit_cutoff = broadcast_overfit_cutoff,
            )
        
        # Baseline check - no of incorrect pred baselines
        if broadcast_baseline_checks_act_pred_needed:
            udf_output_data = baseline_checks_act_pred(
                df = udf_output_data, 
                actual_col = broadcast_actual_col, 
                predicted_base_col = broadcast_predicted_base_col, 
                tpr_col = broadcast_tpr_col,
                no_of_inc_base_cutoff = broadcast_no_of_inc_base_cutoff,
            )
        
        # WoW Baseline check
        if broadcast_baseline_check_mape_needed:
            udf_output_data = baseline_check_mape(
                df = udf_output_data, 
                actual_col = broadcast_actual_col, 
                predicted_base_col = broadcast_predicted_base_col, 
                tpr_col = broadcast_tpr_col, 
                baseline_mape_cutoff = broadcast_baseline_mape_cutoff)
        
        udf_output_data["status"] = "success"

        return udf_output_data
    
    except Exception as e:
        udf_output_data = udf_input_data.copy()

        if broadcast_train_test_rsq_mape_check_needed:
            udf_output_data["train_rsq_flag"] = -1.0
            udf_output_data["test_rsq_flag"] = -1.0
            udf_output_data["train_wmape_flag"] = -1.0
            udf_output_data["test_wmape_flag"] = -1.0
            udf_output_data["overfitting_model"] = -1.0

        if broadcast_rsq_mape_check_needed:
            udf_output_data["rsq_flag"] = -1.0
            udf_output_data["wmape_flag"] = -1.0

        if broadcast_baseline_checks_act_pred_needed:
            udf_output_data["no_of_incorrect_base"] = -1.0
            udf_output_data['too_many_incorrect_base'] = -1.0
            udf_output_data['cut_off_no_of_weeks'] = -1.0

        if broadcast_baseline_check_mape_needed:
            udf_output_data["high_base_MAPE"] = -1.0
            udf_output_data['base_MAPE'] = -1.0

        udf_output_data["status"] = str(traceback.format_exc()) 
        
        return udf_output_data


In [0]:
pd_df = contrib_data.filter(F.col("F_CODE") == "0307").to_pandas()
pd_df.shape

In [ ]:
qc_modules_udf(pd_df)

## 6.6. UDF call

In [0]:
results_s = (
    contrib_data
    .groupBy(['ITERATION_ID'])
    .applyInPandas(qc_modules_udf, output_schema=output_data_schema)
)

In [0]:
results_s

## 6.7. Export QC results

In [0]:
results_s_with_ts = results_s.with_column("LOAD_TS", F.current_timestamp())
results_s_with_ts.write.mode("overwrite").save_as_table(intermediate_table_name)
print(results_s_with_ts.count())

In [ ]:
test_df = session.table(intermediate_table_name)
print("All data ->", test_df.count())
latest_ts = test_df.select(F.max("LOAD_TS")).collect()[0][0]
results_s = test_df.filter(F.col("LOAD_TS") == F.lit(latest_ts))
print("Latest data ->", results_s.count())

In [ ]:
results_s = results_s.with_column(
    "MODEL_IDENTIFIER",
    F.concat_ws(F.lit("_"), *[F.col(c) for c in modeling_granularity])
)
currect_upcs = [row[0] for row in results_s.select("MODEL_IDENTIFIER").distinct().collect()]
len(currect_upcs)

In [ ]:
df_data = df_data.with_column(
    "MODEL_IDENTIFIER",
    F.concat_ws(F.lit("_"), *[F.col(c) for c in modeling_granularity])
)

In [0]:
# 1. Group by Product UPC and sum POSLocalCurrencyAmount
sales_share_df = df_data.groupBy(["MODEL_IDENTIFIER"]) \
    .agg(F.sum(actual_col).alias(actual_col))

# 2. Compute total sum for denominator
total_sum = sales_share_df.agg(F.sum(actual_col).alias("TOTAL")).collect()[0]["TOTAL"]

# 3. Compute sales share
sales_share_df = sales_share_df.withColumn(
    "SALES_SHARE",
    F.col(actual_col) / F.lit(total_sum)
)

# 4. Drop POSLocalCurrencyAmount column
sales_share_df = sales_share_df.drop(actual_col)
sales_share_df.select("SALES_SHARE").agg(F.sum("SALES_SHARE")).show()

In [0]:
sales_share_sum = (
    sales_share_df
    .filter(F.col("MODEL_IDENTIFIER").isin(currect_upcs))
    .agg(F.sum("SALES_SHARE").alias("TOTAL_SALES_SHARE"))
    .collect()[0]["TOTAL_SALES_SHARE"]
)
print(sales_share_sum)

# 7. Data Summary

## 7.1. Criteria 1
Train WMAPE within threshold, No overfitting

In [ ]:
# 1. UPCs where train_wmape_flag == 0
upc_train = (
    results_s
    .filter(F.col("TRAIN_WMAPE_FLAG") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# 2. UPCs where overfitting_model == 0
upc_overfit = (
    results_s
    .filter(F.col("OVERFITTING_MODEL") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# 3. Intersection of the two sets
test_rsq_cut_df = upc_train.intersect(upc_overfit)

# 4. Length of intersection
test_rsq_cut_len = test_rsq_cut_df.count()

# 5. Sum of sales share for those UPCs
sales_share_sum = (
    sales_share_df
    .join(test_rsq_cut_df, on="MODEL_IDENTIFIER", how="inner")
    .agg(F.sum("SALES_SHARE").alias("TOTAL_SALES_SHARE"))
    .collect()[0]["TOTAL_SALES_SHARE"]
)

(test_rsq_cut_len, sales_share_sum)

## 7.2. Criteria 2
 Train & test WMAPE within threshold, No overfitting

In [0]:
# 1. UPCs where train_wmape_flag == 0
upc_train = (
    results_s
    .filter(F.col("TRAIN_WMAPE_FLAG") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# 2. UPCs where overfitting_model == 0
upc_overfit = (
    results_s
    .filter(F.col("OVERFITTING_MODEL") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# UPCs where test_wmape_flag == 0
upc_test = (
    results_s
    .filter(F.col("TEST_WMAPE_FLAG") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# Intersection of all three
test_wmape_cut_df = upc_train.intersect(upc_overfit).intersect(upc_test)

# Count of UPCs
test_wmape_cut_len = test_wmape_cut_df.count()

# Sum of sales share for these UPCs
sales_share_sum = (
    sales_share_df
    .join(test_wmape_cut_df, on="MODEL_IDENTIFIER", how="inner")
    .agg(F.sum("SALES_SHARE").alias("TOTAL_SALES_SHARE"))
    .collect()[0]["TOTAL_SALES_SHARE"]
)

(test_wmape_cut_len, sales_share_sum)

## 7.3. Criteria 3

In [0]:
# 1. UPCs where train_wmape_flag == 0
upc_train = (
    results_s
    .filter(F.col("TRAIN_WMAPE_FLAG") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# 2. UPCs where overfitting_model == 0
upc_overfit = (
    results_s
    .filter(F.col("OVERFITTING_MODEL") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# UPCs where test_wmape_flag == 0
upc_test = (
    results_s
    .filter(F.col("TEST_WMAPE_FLAG") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)


# UPCs where test_wmape_flag == 0
upc_base_mape = (
    results_s
    .filter(F.col("HIGH_BASE_MAPE") == 0)
    .select("MODEL_IDENTIFIER")
    .distinct()
)


# Intersection of all three
base_wmape_cut_df = upc_train.intersect(upc_overfit).intersect(upc_test).intersect(upc_base_mape)

# Count of UPCs
base_wmape_cut_len = base_wmape_cut_df.count()

# Sum of sales share for these UPCs
sales_share_sum = (
    sales_share_df
    .join(base_wmape_cut_df, on="MODEL_IDENTIFIER", how="inner")
    .agg(F.sum("SALES_SHARE").alias("TOTAL_SALES_SHARE"))
    .collect()[0]["TOTAL_SALES_SHARE"]
)

(base_wmape_cut_len, sales_share_sum)

## 7.4. Extracting the best model iterations
  For each F_CODE, if an iteration satisfy all the QC criteria, then the Hyperparameter combination of the iteration is considered as the final model. If all the iterations for a F_CODE fails the QC criteria, then the final model is selected based on test WMAPE

In [0]:
# Step 1: Filter Iterations
list_of_itr_df = (
    results_s
    .filter(
        (F.col("TRAIN_WMAPE_FLAG") == 0) &
        (F.col("TEST_WMAPE_FLAG") == 0) &
        (F.col("OVERFITTING_MODEL") == 0) &
        (F.col("HIGH_BASE_MAPE") == 0)
    )
    .select("ITERATION_ID")
    .distinct()
)

# Step 2: Find list of Products corresponding to these Iterations
list_prods_df = (
    results_s
    .join(list_of_itr_df, on="ITERATION_ID", how="inner")
    .select("MODEL_IDENTIFIER")
    .distinct()
)

# Step 3: Sales share for these products
sales_share_sum = (
    sales_share_df
    .join(list_prods_df, on="MODEL_IDENTIFIER", how="inner")
    .agg(F.sum("SALES_SHARE").alias("TOTAL_SALES_SHARE"))
    .collect()[0]["TOTAL_SALES_SHARE"]
)
print("sales share", sales_share_sum)

# Step 4: Filter dataset with valid iterations
results_s_df_filtered = results_s.join(list_of_itr_df, on="ITERATION_ID", how="inner")

# Count products
num_products_1 = results_s_df_filtered.select("MODEL_IDENTIFIER").distinct().count()
print("no of products", num_products_1)

# Step 5: Compute min base_MAPE per Product UPC
min_base_mape_itr = (
    results_s_df_filtered
    .groupBy("MODEL_IDENTIFIER")
    .agg(F.min("BASE_MAPE").alias("MIN_BASE_MAPE"))
)

# Step 6: Join back and keep only rows where base_MAPE == min_base_MAPE
results_s_df_filtered = (
    results_s_df_filtered
    .join(min_base_mape_itr, on="MODEL_IDENTIFIER", how="inner")
    .filter(F.col("BASE_MAPE") == F.col("MIN_BASE_MAPE"))
)

# Count products again
num_products_2 = results_s_df_filtered.select("MODEL_IDENTIFIER").distinct().count()
print("no of products", num_products_2)

# Step 7: Final Iterations per Product UPC (take min Iteration_ID)
final_iterations_df = (
    results_s_df_filtered
    .groupBy("MODEL_IDENTIFIER")
    .agg(F.min("Iteration_ID").alias("Iteration_ID"))
)

In [ ]:
results_s_df_filtered

In [0]:
final_iterations_df

# 8. Remaining Combinations

In [0]:
# Step 1: Exclude products that are in list_prods_df
rem_prods_df = (
    results_s
    .join(list_prods_df, on="MODEL_IDENTIFIER", how="left_anti")  # keeps only rows NOT in list_prods_df
)

# Step 2: Group by Product UPC and count distinct Iteration_ID
rem_prods_grouped = (
    rem_prods_df
    .groupBy("MODEL_IDENTIFIER")
    .agg(F.countDistinct("ITERATION_ID").alias("nunique_Iteration_ID"))
)

# To preview
rem_prods_grouped

In [0]:
# Step 1: Exclude products in list_prods
rem_prods_df = results_s.join(list_prods_df, on="MODEL_IDENTIFIER", how="left_anti")

# Step 2: For each Product UPC, find min base_MAPE
selected_itr_df = (
    rem_prods_df
    .groupBy("MODEL_IDENTIFIER")
    .agg(F.min("TEST_WMAPE").alias("MIN_TEST_WMAPE"))
)

# Step 3: Join back to filter only rows with min_base_MAPE
rem_prods_df = rem_prods_df.join(selected_itr_df, on="MODEL_IDENTIFIER", how="inner")
rem_prods_df = rem_prods_df.filter(F.col("TEST_WMAPE") == F.col("MIN_TEST_WMAPE"))

# Step 4: Select distinct Product UPC & Iteration_ID
rem_prods_itr_final = rem_prods_df.select("MODEL_IDENTIFIER", "ITERATION_ID").dropDuplicates()

# Step 5: For each Product UPC, pick max Iteration_ID
rem_prods_itr_final = (
    rem_prods_itr_final
    .groupBy("MODEL_IDENTIFIER")
    .agg(F.max("ITERATION_ID").alias("ITERATION_ID"))
)

# Final preview
rem_prods_itr_final

In [ ]:
final_iterations_df = final_iterations_df.unionByName(rem_prods_itr_final)
final_iterations_df

In [ ]:
# Step 1: Filter Iterations
list_of_itr_df = (
    final_iterations_df
    .select("ITERATION_ID")
    .distinct()
)

# Step 2: Find list of Products corresponding to these Iterations
final_cont_df = (
    results_s
    .join(list_of_itr_df, on="ITERATION_ID", how="inner")
    
)

# Step 2: Find list of Products corresponding to these Iterations
final_coeff_df = (
    model_coeff_df
    .join(list_of_itr_df, on="ITERATION_ID", how="inner")
    
)

In [ ]:
final_cont_df

In [ ]:
final_coeff_df

In [ ]:
results_s_with_ts = final_coeff_df.with_column("LOAD_TS", F.current_timestamp())
results_s_with_ts.write.mode("overwrite").save_as_table(coef_output_table_name)
print(results_s_with_ts.count())

In [ ]:
results_s_with_ts = final_cont_df.with_column("LOAD_TS", F.current_timestamp())
results_s_with_ts.write.mode("overwrite").save_as_table(cont_output_table_name)
print(results_s_with_ts.count())